# 04 - Ticket resolution workflow (approve with edits)

**Scenario.** An L2 support lead reviews open **Zentrix Z9 Plus** tickets about *slow charging with the supplied
adapter*, wants the manual's guidance, and asks to **resolve TKT-000027** with that guidance.

**You will see** ticket search (partial, case-insensitive), the manual lookup, a proposed `resolve_ticket`, and an
approval where the human **edits** an argument before execution.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, `JUDGE_OPENAI_API_KEY` and `SERPER_API_KEY`; the vector stores (`make ingest`); the CRM & Support MCP server (`make mcp-server-up`).

After one run the ticket is resolved; `make mcp-seed-reset` restores the demo data.

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
# Start the multi-agent service: SQLite memory + CRM MCP tools + 5 specialists + Coordinator graph
from agentic_rag.graph import CoordinatorService

service = await CoordinatorService.create(settings)
print(service.crm.describe())
print(service.roster())

In [ ]:
QUESTION = ("List the open Zentrix Z9 Plus tickets about slow charging with the supplied adapter, check the manual "
            "for the recommended fix, and resolve TKT-000027 with that guidance. Resolved by: Support Desk.")

result = await service.ask(QUESTION, user_id="l2-lead-meera", on_event=print)
result

## Approve with an edit
Decisions can be given per action: `approve`, `reject` or `edit` with replacement `arguments`.

In [ ]:
import json
if result.status == "needs_approval":
    print(json.dumps(result.interrupt["actions"], indent=2)[:2500])
    decisions = [{"id": a["id"], "decision": "edit", "arguments": {"resolved_by": "Meera (L2 Support)"}}
                 if a["tool"] == "resolve_ticket" else {"id": a["id"], "decision": "reject"}
                 for a in result.interrupt["actions"]]
    result = await service.resume(result.thread_id, {"decisions": decisions}, on_event=print)
result

In [ ]:
for action in result.actions:
    print(action["decision"].upper(), action["tool"], json.dumps(action["arguments"])[:300])

In [ ]:
# Close the memory databases
await service.aclose()

## What to observe
- The ticket search uses partial matching (`slow charging`) and a status filter.
- The resolution remarks come from the manual evidence; the human changed `resolved_by` before execution.
- Resolving an already resolved ticket is refused by the server's business rules (see notebook 10).